In [ ]:
%pip install db-dtypes

In [ ]:
%pip uninstall torch -y
%pip install torch==2.1.0+cpu torchvision==0.16.0+cpu torchaudio==2.1.0+cpu -f https://download.pytorch.org/whl/torch_stable.html


In [14]:
%pip install google-cloud-bigquery-storage


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip uninstall torch
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gh-masterthesis-jasmine-29e28ba279bc.json"
from google.cloud import bigquery

# Create BigQuery client
client = bigquery.Client()
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder


- load SQL extend function

In [ ]:
%load_ext sql

- connect to SQLite database (.db)

In [2]:
%sql sqlite:///rxjs-ghtorrent.db

In [3]:
# Load the SQLite database
conn = sqlite3.connect("C:/Users/user/Downloads/Master thesis/ghelephant/ghelephant-main/rxjs-ghtorrent.db")  # change this to your actual path


In [4]:
# Show all available tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)


                     name
0             schema_info
1         sqlite_sequence
2          commit_parents
3    organization_members
4         project_members
5    pull_request_commits
6         project_commits
7            issue_labels
8           pull_requests
9                projects
10                commits
11              followers
12        commit_comments
13   pull_request_history
14  pull_request_comments
15                 issues
16           issue_events
17         issue_comments
18            repo_labels
19        repo_milestones
20               watchers
21      project_languages
22                  users


In [17]:
query = "SELECT * FROM watchers LIMIT 10000;"
df = pd.read_sql_query(query, conn)
print(df)


      repo_id  user_id                  created_at
0          12      159  2013-02-10 07:48:59.000000
1          14      160  2013-08-06 05:14:54.000000
2          14      161  2013-11-07 07:23:07.000000
3          14      162  2013-10-27 13:20:46.000000
4          14      163  2012-07-04 19:42:09.000000
...       ...      ...                         ...
5105        1     5925  2011-04-21 14:23:06.000000
5106        1     5926  2015-03-27 02:57:45.000000
5107        1     5927  2014-07-08 12:48:55.000000
5108        1     5928  2013-05-06 15:45:50.000000
5109        1     5929  2013-03-19 03:30:33.000000

[5110 rows x 3 columns]


In [ ]:
query = """
SELECT user_id, repo_id AS project_id, 'star' AS interaction_type, created_at
FROM watchers
"""
stars_df = pd.read_sql_query(query, conn)
stars_df


In [ ]:
# sql
query = """
SELECT 
  user_id AS developer_id,
  repo_id AS project_id,
  created_at AS timestamp,
  'star' AS interaction_type
FROM 
  `ghtorrentmysql1906.MySQL1906.watchers`
WHERE DATE(created_at) BETWEEN '2008-01-01' AND '2015-12-31'
"""
# search
query_job = client.query(query)

# get the result
results_0812 = query_job.result().to_dataframe()

# 看看結果
print(results_0812.head())


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1933: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [11]:
results_0812

,developer_id,project_id,timestamp,interaction_type
0,32455,927437,2012-01-29 10:43:58+00:00,star
1,934253,8797901,2014-03-27 13:50:43+00:00,star
2,5956845,5613005,2015-11-05 04:33:14+00:00,star
3,746527,13453519,2015-10-15 13:39:52+00:00,star
4,958290,1077711,2014-04-23 16:58:28+00:00,star
...,...,...,...,...
36119934,2614960,2112,2013-09-25 11:26:54+00:00,star
36119935,14270,2880,2009-09-07 21:27:53+00:00,star
36119936,6637678,22171969,2015-07-16 13:18:23+00:00,star
36119937,2975222,21975873,2015-08-19 06:24:54+00:00,star


In [10]:
results_0815 = results_0812
results_0815

,developer_id,project_id,timestamp,interaction_type
0,32455,927437,2012-01-29 10:43:58+00:00,star
1,934253,8797901,2014-03-27 13:50:43+00:00,star
2,5956845,5613005,2015-11-05 04:33:14+00:00,star
3,746527,13453519,2015-10-15 13:39:52+00:00,star
4,958290,1077711,2014-04-23 16:58:28+00:00,star
...,...,...,...,...
36119934,2614960,2112,2013-09-25 11:26:54+00:00,star
36119935,14270,2880,2009-09-07 21:27:53+00:00,star
36119936,6637678,22171969,2015-07-16 13:18:23+00:00,star
36119937,2975222,21975873,2015-08-19 06:24:54+00:00,star


In [14]:
results_0815.to_csv('results_2008to2015.csv', index=False)

In [5]:


query = """
SELECT u.login AS user, p.name AS project
FROM `ghtorrentmysql1906.MySQL1906.watchers` w
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON w.user_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON w.repo_id = p.id
LIMIT 3000
"""

df = client.query(query).to_dataframe()
print(df.head())


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1933: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


        user          project
0   faith-hb  Folding-Android
1  R3FL3CT0R          kubefwd
2      viplz  PLMCodeTemplate
3    zzy7584         Mind-Map
4       fith        phpdoc-md


- Time range for the database

| earliest | latest |
| -------- | ------ |
|2007-10-29 13:37:16 UTC| 2019-05-31 23:54:55 UTC|

- issues 

| earliest | latest |
| -------- | ------ |
|1970-01-02 00:00:00 UTC|2019-05-31 23:55:16 UTC|

🧩 合併成完整互動資料集（Advanced！）

In [ ]:
query = """
-- Watch 行為
SELECT u.login AS user, p.name AS project, user_id AS developer_id, repo_id AS project_id, 'watch' AS interaction_type, created_at
FROM `ghtorrentmysql1906.MySQL1906.watchers`
WHERE EXTRACT(YEAR FROM created_at) BETWEEN 2008 AND 2015

UNION ALL

-- 建立 pull_request 開啟的行為
SELECT
  prh.actor_id AS developer_id,
  pr.base_repo_id AS project_id,
  'pull_request' AS interaction_type,
  prh.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.pull_request_history` prh
JOIN `ghtorrentmysql1906.MySQL1906.pull_requests` pr
  ON prh.pull_request_id = pr.id
WHERE prh.action = 'opened'
  AND EXTRACT(YEAR FROM prh.created_at) BETWEEN 2008 AND 2015

UNION ALL

-- 建立 pull_request 留言的行為
SELECT
  prc.user_id AS developer_id,
  pr.base_repo_id AS project_id,
  'pull_comment' AS interaction_type,
  prc.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.pull_request_comments` prc
JOIN `ghtorrentmysql1906.MySQL1906.pull_requests` pr
  ON prc.pull_request_id = pr.id
WHERE EXTRACT(YEAR FROM prc.created_at) BETWEEN 2008 AND 2015

"""

# 執行查詢並將結果轉換為 DataFrame
react_1 = client.query(query).to_dataframe()

# 輸出結果查看
print(react_1.head())


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1933: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   developer_id  project_id interaction_type                created_at
0        136914    18075206     pull_comment 2015-05-22 08:24:34+00:00
1       5785366    12794178     pull_comment 2015-11-17 08:15:14+00:00
2         17950    13413311     pull_comment 2015-05-05 14:18:44+00:00
3         60802       13721     pull_comment 2014-05-19 11:27:17+00:00
4         71341       10084     pull_comment 2015-12-14 13:13:12+00:00


In [24]:
react_1

,developer_id,project_id,interaction_type,created_at
0,136914,18075206,pull_comment,2015-05-22 08:24:34+00:00
1,5785366,12794178,pull_comment,2015-11-17 08:15:14+00:00
2,17950,13413311,pull_comment,2015-05-05 14:18:44+00:00
3,60802,13721,pull_comment,2014-05-19 11:27:17+00:00
4,71341,10084,pull_comment,2015-12-14 13:13:12+00:00
...,...,...,...,...
52992882,2902425,11745601,watch,2014-10-21 04:22:15+00:00
52992883,2563731,20462657,watch,2015-06-19 00:58:05+00:00
52992884,2870581,4013121,watch,2015-01-15 06:29:47+00:00
52992885,2442869,5783361,watch,2013-10-14 19:25:18+00:00


In [36]:
query = """
-- Watch 行為
SELECT user_id AS developer_id, repo_id AS project_id, 'watch' AS interaction_type, created_at
FROM `ghtorrentmysql1906.MySQL1906.watchers`
WHERE EXTRACT(YEAR FROM created_at) BETWEEN 2008 AND 2015

UNION ALL

-- 建立 pull_request 開啟的行為
SELECT
  prh.actor_id AS developer_id,
  pr.base_repo_id AS project_id,
  'pull_request' AS interaction_type,
  prh.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.pull_request_history` prh
JOIN `ghtorrentmysql1906.MySQL1906.pull_requests` pr
  ON prh.pull_request_id = pr.id
WHERE prh.action = 'opened'
  AND EXTRACT(YEAR FROM prh.created_at) BETWEEN 2016 AND 2018

UNION ALL

-- 建立 pull_request 留言的行為
SELECT
  prc.user_id AS developer_id,
  pr.base_repo_id AS project_id,
  'pull_comment' AS interaction_type,
  prc.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.pull_request_comments` prc
JOIN `ghtorrentmysql1906.MySQL1906.pull_requests` pr
  ON prc.pull_request_id = pr.id
WHERE EXTRACT(YEAR FROM prc.created_at) BETWEEN 2008 AND 2015

"""

# 執行查詢並將結果轉換為 DataFrame
validation = client.query(query).to_dataframe()

# 輸出結果查看
print(validation.head())


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1933: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   developer_id  project_id interaction_type                created_at
0        198738    12433305     pull_comment 2014-10-08 21:39:34+00:00
1        572655    10230767     pull_comment 2015-11-13 19:00:00+00:00
2         50307    11372864     pull_comment 2015-09-18 18:28:49+00:00
3        111358       40291     pull_comment 2014-08-06 07:01:05+00:00
4       5023683    13469973     pull_comment 2015-07-07 18:27:10+00:00


In [37]:
validation

,developer_id,project_id,interaction_type,created_at
0,198738,12433305,pull_comment,2014-10-08 21:39:34+00:00
1,572655,10230767,pull_comment,2015-11-13 19:00:00+00:00
2,50307,11372864,pull_comment,2015-09-18 18:28:49+00:00
3,111358,40291,pull_comment,2014-08-06 07:01:05+00:00
4,5023683,13469973,pull_comment,2015-07-07 18:27:10+00:00
...,...,...,...,...
76590095,35544893,63237305,pull_request,2017-06-15 12:35:05+00:00
76590096,4199854,13494281,pull_request,2017-03-28 07:41:28+00:00
76590097,36535684,110906376,pull_request,2018-10-11 22:55:52+00:00
76590098,619663,7599626,pull_request,2017-03-19 08:45:21+00:00


In [41]:
validation.to_parquet("validation_2016to2018.parquet", index=False)

In [ ]:
validation = pd.read_parquet("validation_2016to2018.parquet")


- under sample `oss_interaction_sample2.parquet`
result of label with 0 and 1<br>

- save as parquet and read

In [27]:
react_1.to_parquet("oss_interact_2008to2015.parquet", index=False)


In [4]:
#oss_interact = pd.read_parquet("oss_interact_2008to2015.parquet")
oss_interact = pd.read_parquet("oss_interaction_sample2.parquet")
oss_interact


,user,project,developer_id,project_id,interaction_type,interaction_time
0,conceptualitis,---,209651,1967344,star,2013-01-14 23:44:41+00:00
1,maheshkurmi,-Android-Super-Mario,995518,13022690,star,2015-12-29 10:54:46+00:00
2,liuguangli,-AndroidBaseProject,8100341,25526052,star,2015-11-20 07:16:34+00:00
3,FedBack,-FedBack-master,317875,340882,star,2012-07-08 14:20:02+00:00
4,radovankavicky,-No-longer-active-R-Finance,7778816,18866549,star,2015-04-16 09:45:17+00:00
...,...,...,...,...,...,...
349995,JakeHartnell,Noura,8436368,99830,follow,2011-01-23 13:00:53+00:00
349996,alvations,Nourahussein,2390544,10887797,follow,2015-12-01 17:39:54+00:00
349997,RX14,NoviantoEkoBudiman,998314,10557586,follow,2015-11-17 23:45:37+00:00
349998,huntexD,Novium,28560483,2152584,follow,2013-11-06 12:38:24+00:00


In [ ]:
val_df = validation.loc[:100000]


# 你可以只取有互動的行為，設 label=1
val_df = val_df[val_df["interaction_type"].isin(["watch", "pull_request"])]
val_df["label"] = 1  # 全是正樣本

# 然後像 training 一樣也負取樣：
all_users = val_df["developer_id"].unique()
all_items = val_df["project_id"].unique()

positive_pairs = set(zip(val_df["developer_id"], val_df["project_id"]))
num_negatives = len(val_df)

negatives = []
np.random.seed(42)
while len(negatives) < num_negatives:
    user = np.random.choice(all_users)
    item = np.random.choice(all_items)
    if (user, item) not in positive_pairs:
        negatives.append((user, item))

neg_val_df = pd.DataFrame(negatives, columns=["developer_id", "project_id"])
neg_val_df["label"] = 0

val_df = pd.concat([val_df[["developer_id", "project_id", "label"]], neg_val_df])

val_dataset = InteractionDataset(val_users, val_items, val_labels)
val_loader = DataLoader(val_dataset, batch_size=128)

# 評估模型準確率 / AUC 等
model.eval()
all_preds = []
all_truth = []

with torch.no_grad():
    for user, item, label in val_loader:
        user = user.to(device)
        item = item.to(device)
        preds = model(user, item).cpu().numpy()
        all_preds.extend(preds)
        all_truth.extend(label.numpy())

from sklearn.metrics import roc_auc_score, accuracy_score

C:\Users\user\AppData\Local\Temp\ipykernel_15424\2157061896.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df["user_idx"] = user_encoder.fit_transform(val_df["developer_id"])
C:\Users\user\AppData\Local\Temp\ipykernel_15424\2157061896.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df["item_idx"] = user_encoder.fit_transform(val_df["project_id"])
C:\Users\user\AppData\Local\Temp\ipykernel_15424\2157061896.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

,developer_id,project_id,interaction_type,created_at,user_idx,item_idx,label
0,198738,12433305,pull_comment,2014-10-08 21:39:34+00:00,8705,11387,1
1,572655,10230767,pull_comment,2015-11-13 19:00:00+00:00,13223,9596,1
2,50307,11372864,pull_comment,2015-09-18 18:28:49+00:00,3833,10620,1
3,111358,40291,pull_comment,2014-08-06 07:01:05+00:00,6375,1263,1
4,5023683,13469973,pull_comment,2015-07-07 18:27:10+00:00,28719,12146,1
...,...,...,...,...,...,...,...
99996,3775124,40934325,pull_request,2018-08-20 14:22:43+00:00,26477,26095,1
99997,2783109,82295454,pull_request,2018-02-07 03:50:46+00:00,23435,41741,1
99998,41509583,117270247,pull_request,2018-12-05 10:24:56+00:00,53057,51853,1
99999,44717270,103270440,pull_request,2018-07-31 09:49:07+00:00,54879,48157,1


In [53]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
val_labels = encoder.fit_transform(val_df["interaction_type"])
val_df


,developer_id,project_id,interaction_type,created_at,user_idx,item_idx,label
0,198738,12433305,pull_comment,2014-10-08 21:39:34+00:00,8705,11387,1
1,572655,10230767,pull_comment,2015-11-13 19:00:00+00:00,13223,9596,1
2,50307,11372864,pull_comment,2015-09-18 18:28:49+00:00,3833,10620,1
3,111358,40291,pull_comment,2014-08-06 07:01:05+00:00,6375,1263,1
4,5023683,13469973,pull_comment,2015-07-07 18:27:10+00:00,28719,12146,1
...,...,...,...,...,...,...,...
99996,3775124,40934325,pull_request,2018-08-20 14:22:43+00:00,26477,26095,1
99997,2783109,82295454,pull_request,2018-02-07 03:50:46+00:00,23435,41741,1
99998,41509583,117270247,pull_request,2018-12-05 10:24:56+00:00,53057,51853,1
99999,44717270,103270440,pull_request,2018-07-31 09:49:07+00:00,54879,48157,1


In [28]:
test = oss_interact.loc[:1000000]
test

,developer_id,project_id,interaction_type,created_at,user_idx,item_idx,label
0,136914,18075206,pull_comment,2015-05-22 08:24:34+00:00,106540,2864968,1
1,5785366,12794178,pull_comment,2015-11-17 08:15:14+00:00,1545278,2294717,1
2,17950,13413311,pull_comment,2015-05-05 14:18:44+00:00,15582,2355245,1
3,60802,13721,pull_comment,2014-05-19 11:27:17+00:00,48448,9940,1
4,71341,10084,pull_comment,2015-12-14 13:13:12+00:00,56703,7325,1
...,...,...,...,...,...,...,...
999996,150750,8873349,watch,2015-03-23 00:51:29+00:00,116501,1743210,1
999997,37061,7432069,watch,2014-01-15 20:07:04+00:00,31486,1545956,1
999998,658968,9529989,watch,2015-08-19 01:44:48+00:00,373450,1830394,1
999999,701300,20409221,watch,2015-05-28 02:53:35+00:00,387372,3088277,1


In [42]:
# Encode developer_id and project_id
from sklearn.preprocessing import LabelEncoder
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

oss_interact["user_idx"] = user_encoder.fit_transform(oss_interact["developer_id"])
oss_interact["item_idx"] = item_encoder.fit_transform(oss_interact["project_id"])

user_count = oss_interact['user_idx'].nunique()
item_count = oss_interact['item_idx'].nunique()
rate_count = 2  # We will use binary interactions (1 = interacted, 0 = negative sample)

interaction_map = {
    "star": 0,            # 輕度 → 表示感興趣
    "follow": 1,          # 關注某人，代表信任
    "fork": 2,            # 想進一步探索或修改
    "issue": 3,           # 主動參與討論，提出需求
    "pull_request": 4,    # 真正貢獻代碼 → 積極參與
    "pull_comment": 5,    # 對貢獻作回饋 → 團隊互動強
    "commit": 6           # 真實寫入代碼 → 最高參與度
}
oss_interact["rating"] = oss_interact["interaction_type"].map(interaction_map)
oss_interact

,user,project,developer_id,project_id,interaction_type,interaction_time,user_idx,item_idx,rating,label
0,conceptualitis,---,209651,1967344,star,2013-01-14 23:44:41+00:00,41633,44875,0,1
1,maheshkurmi,-Android-Super-Mario,995518,13022690,star,2015-12-29 10:54:46+00:00,83513,137650,0,1
2,liuguangli,-AndroidBaseProject,8100341,25526052,star,2015-11-20 07:16:34+00:00,204507,183895,0,1
3,FedBack,-FedBack-master,317875,340882,star,2012-07-08 14:20:02+00:00,52250,24065,0,1
4,radovankavicky,-No-longer-active-R-Finance,7778816,18866549,star,2015-04-16 09:45:17+00:00,201435,164012,0,1
...,...,...,...,...,...,...,...,...,...,...
349995,JakeHartnell,Noura,8436368,99830,follow,2011-01-23 13:00:53+00:00,207503,13115,1,1
349996,alvations,Nourahussein,2390544,10887797,follow,2015-12-01 17:39:54+00:00,126569,122879,1,1
349997,RX14,NoviantoEkoBudiman,998314,10557586,follow,2015-11-17 23:45:37+00:00,83628,120358,1,1
349998,huntexD,Novium,28560483,2152584,follow,2013-11-06 12:38:24+00:00,222590,46057,1,1


In [ ]:
from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

test["user_idx"] = user_encoder.fit_transform(test["developer_id"])
test["item_idx"] = user_encoder.fit_transform(test["project_id"])

In [30]:
# interact label
test["label"] = 1


# 所有使用者與專案集合
all_users = test["user_idx"].unique()
all_items = test["item_idx"].unique()

# 建立 (user, item) 正樣本集合
positive_pairs = set(zip(test["user_idx"], test["item_idx"]))

# 負樣本數與正樣本一樣
num_negatives = len(test)
negatives = []

np.random.seed(42)
while len(negatives) < num_negatives:
    user = np.random.choice(all_users)
    item = np.random.choice(all_items)
    if (user, item) not in positive_pairs:
        negatives.append((user, item))

neg_df = pd.DataFrame(negatives, columns=["user_idx", "item_idx"])
neg_df["label"] = 0


C:\Users\user\AppData\Local\Temp\ipykernel_15424\1382232808.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["label"] = 1


In [23]:
# train_df = pd.concat([
#     oss_interact[["user_idx", "item_idx", "rating"]],
#     neg_df
# ], ignore_index=True)
train_users = oss_interact["user_idx"].values
train_items = oss_interact["item_idx"].values
train_labels = oss_interact["rating"].values



In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


CUDA available: True
Device: NVIDIA GeForce MX330


In [36]:
# interact label
oss_interact["label"] = 1


# 所有使用者與專案集合
all_users = oss_interact["user_idx"].unique()
all_items = oss_interact["item_idx"].unique()

# 建立 (user, item) 正樣本集合
positive_pairs = set(zip(oss_interact["user_idx"], oss_interact["item_idx"]))

# 負樣本數與正樣本一樣
num_negatives = len(oss_interact)
negatives = []

np.random.seed(42)
while len(negatives) < num_negatives:
    user = np.random.choice(all_users)
    item = np.random.choice(all_items)
    if (user, item) not in positive_pairs:
        negatives.append((user, item))

neg_df = pd.DataFrame(negatives, columns=["user_idx", "item_idx"])
neg_df["label"] = 0


In [28]:
train_df = pd.concat([
    oss_interact[["user_idx", "item_idx", "label"]],
    neg_df
], ignore_index=True)
train_users = train_df["user_idx"].values
train_items = train_df["item_idx"].values
train_labels = train_df["label"].values



In [46]:
# Set device
device = torch.device("cpu")
print("Using device:", device)


Using device: cpu


✅ NCF model definition

In [47]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=32):
        super(NCF, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim * 2, 64)
        self.fc2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)

    def forward(self, user_ids, item_ids):
        user_embed = self.user_embedding(user_ids)
        item_embed = self.item_embedding(item_ids)
        x = torch.cat([user_embed, item_embed], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.output(x)).squeeze()  # [B] -> scalar
        return x

# Dataset class
class InteractionDataset(Dataset):
    def __init__(self, users, items, labels):
        self.users = users
        self.items = items
        self.labels = labels

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.users[idx], dtype=torch.long),
            torch.tensor(self.items[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float),
        )

In [48]:
# Get user/item counts (make sure these vars exist)
num_users = train_users.max() + 1
num_items = train_items.max() + 1

# Initialize model on device
model = NCF(num_users, num_items).to(device)

- FIRST MODEL with label 0 and 1

In [31]:
# Optimizer & loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

# Prepare DataLoader
train_dataset = InteractionDataset(train_users, train_items, train_labels)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Training loop
for epoch in range(10):
    total_loss = 0
    model.train()
    for user, item, label in train_loader:
        # Move data to device
        user = user.to(device)
        item = item.to(device)
        label = label.to(device)

        # Training step
        optimizer.zero_grad()
        prediction = model(user, item)
        loss = loss_fn(prediction, label)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: Loss {total_loss / len(train_loader):.4f}")

    torch.cuda.empty_cache()


Epoch 1: Loss 0.3066
Epoch 2: Loss 0.2216
Epoch 3: Loss 0.1448
Epoch 4: Loss 0.0884
Epoch 5: Loss 0.0516
Epoch 6: Loss 0.0292
Epoch 7: Loss 0.0164
Epoch 8: Loss 0.0095
Epoch 9: Loss 0.0060
Epoch 10: Loss 0.0040


In [34]:
torch.cuda.empty_cache()


- Validation

In [59]:
val_df = validation.loc[:100000]


# 你可以只取有互動的行為，設 label=1
val_df = val_df[val_df["interaction_type"].isin(["watch", "pull_request"])]
val_df["label"] = 1  # 全是正樣本

# 然後像 training 一樣也負取樣：
all_users = val_df["developer_id"].unique()
all_items = val_df["project_id"].unique()

positive_pairs = set(zip(val_df["developer_id"], val_df["project_id"]))
num_negatives = len(val_df)

negatives = []
np.random.seed(42)
while len(negatives) < num_negatives:
    user = np.random.choice(all_users)
    item = np.random.choice(all_items)
    if (user, item) not in positive_pairs:
        negatives.append((user, item))

neg_val_df = pd.DataFrame(negatives, columns=["developer_id", "project_id"])
neg_val_df["label"] = 0

val_df = pd.concat([val_df[["developer_id", "project_id", "label"]], neg_val_df])


user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

val_df["user_idx"] = user_encoder.fit_transform(val_df["developer_id"])
val_df["item_idx"] = user_encoder.fit_transform(val_df["project_id"])




val_users = val_df["user_idx"].values
val_items = val_df["item_idx"].values
val_labels = val_df["label"].values


val_dataset = InteractionDataset(val_users, val_items, val_labels)
val_loader = DataLoader(val_dataset, batch_size=128)

# 評估模型準確率 / AUC 等
model.eval()
all_preds = []
all_truth = []

with torch.no_grad():
    for user, item, label in val_loader:
        user = user.to(device)
        item = item.to(device)
        preds = model(user, item).cpu().numpy()
        all_preds.extend(preds)
        all_truth.extend(label.numpy())

from sklearn.metrics import roc_auc_score, accuracy_score


print("Validation AUC:", roc_auc_score(all_truth, all_preds))
print("Validation Accuracy:", accuracy_score(all_truth, np.round(all_preds)))

Validation AUC: 0.5089258538651773
Validation Accuracy: 0.5043827312197919


In [56]:
val_dataset = InteractionDataset(val_users, val_items, val_labels)
val_loader = DataLoader(val_dataset, batch_size=256)


In [60]:
def evaluate(model, val_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for user, item, label in val_loader:
            user, item, label = user.to(device), item.to(device), label.to(device)
            pred = model(user, item)
            all_preds.append(pred.cpu())
            all_labels.append(label.cpu())
    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    
    # 計算 Accuracy / Precision / Recall / F1
    predicted = (preds > 0.5).int()
    true = labels.int()
    acc = (predicted == true).float().mean().item()
    
    print(f"Validation Accuracy: {acc:.4f}")

evaluate(model, val_loader)


Validation Accuracy: 0.5044


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(1, predicted)
recall = recall_score(1, predicted)
f1 = f1_score(1, predicted)

print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")


In [ ]:
query = """
-- concate
SELECT user_id, project_id, interaction_type, timestamp FROM (
  -- star
  SELECT
    w.user_id,
    w.repo_id AS project_id,
    'star' AS interaction_type,
    w.created_at AS timestamp
  FROM `ghtorrent-bq.ght.watchers` w
  WHERE w.created_at IS NOT NULL

  UNION ALL

  -- fork
  SELECT
    p.owner_id AS user_id,
    p.forked_from AS project_id,
    'fork' AS interaction_type,
    p.created_at AS timestamp
  FROM `ghtorrent-bq.ght.projects` p
  WHERE p.forked_from IS NOT NULL

  UNION ALL

  -- pull request
  SELECT
    pr.actor_id AS user_id,
    pr.base_repo_id AS project_id,
    'pull_request' AS interaction_type,
    pr.created_at AS timestamp
  FROM `ghtorrent-bq.ght.pull_requests` pr
  WHERE pr.created_at IS NOT NULL

  UNION ALL

  -- issue
  SELECT
    i.reporter_id AS user_id,
    i.repo_id AS project_id,
    'issue' AS interaction_type,
    i.created_at AS timestamp
  FROM `ghtorrent-bq.ght.issues` i
  WHERE i.created_at IS NOT NULL

  UNION ALL

  -- issue comment
  SELECT
    ic.user_id,
    i.repo_id AS project_id,
    'issue_comment' AS interaction_type,
    ic.created_at AS timestamp
  FROM `ghtorrent-bq.ght.issue_comments` ic
  JOIN `ghtorrent-bq.ght.issues` i ON ic.issue_id = i.id
  WHERE ic.created_at IS NOT NULL
)
LIMIT 1000000
"""

react_ = client.query(query).to_dataframe()
print(df.head())

In [ ]:

class NCF(nn.Module):
    def __init__(self, num_users, num_projects, embedding_dim):
        super(NCF, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.project_embedding = nn.Embedding(num_projects, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim * 2, 128)
        self.fc2 = nn.Linear(128, 1)
        
    def forward(self, user, project):
        user_embedded = self.user_embedding(user)
        project_embedded = self.project_embedding(project)
        x = torch.cat([user_embedded, project_embedded], dim=-1)
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

# Initialize the model, loss, and optimizer
model = NCF(num_users=1000, num_projects=500, embedding_dim=64)
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop (just an example)
for epoch in range(100):
    for user, project, label in training_data:  # Assume `training_data` is a dataset of (user, project, interaction)
        optimizer.zero_grad()
        output = model(user, project)
        loss = loss_fn(output, label)
        loss.backward()
        optimizer.step()


In [ ]:
import torch
import torch_geometric
from torch_geometric.nn import GCNConv

class GCNRecommender(nn.Module):
    def __init__(self, num_features, hidden_channels):
        super(GCNRecommender, self).__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)
        return x

# Sample training loop with PyG data
model = GCNRecommender(num_features=64, hidden_channels=128)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop for GCN
for epoch in range(100):
    optimizer.zero_grad()
    out = model(data)  # data is your PyG graph data
    loss = loss_fn(out, labels)
    loss.backward()
    optimizer.step()


#### Get the interaction type data

In [52]:
query = """

WITH interactions AS (
-- Star (same as watch, but included separately if needed)
SELECT 
  u.login AS user,
  p.name AS project,
  w.user_id AS developer_id,
  w.repo_id AS project_id,
  'star' AS interaction_type,
  w.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.watchers` w
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON w.user_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON w.repo_id = p.id
WHERE EXTRACT(YEAR FROM w.created_at) BETWEEN 2010 AND 2015


UNION ALL

-- Fork
SELECT 
  u.login AS user,
  p.name AS project,
  p.owner_id AS developer_id,
  p.forked_from AS project_id,
  'fork' AS interaction_type,
  p.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.projects` p
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON p.owner_id = u.id
WHERE p.forked_from IS NOT NULL
  AND EXTRACT(YEAR FROM p.created_at) BETWEEN 2010 AND 2015

UNION ALL

-- Issue
SELECT 
  u.login AS user,
  p.name AS project,
  i.reporter_id AS developer_id,
  i.repo_id AS project_id,
  'issue' AS interaction_type,
  i.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.issues` i
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON i.reporter_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON i.repo_id = p.id
WHERE EXTRACT(YEAR FROM i.created_at) BETWEEN 2010 AND 2015

UNION ALL

-- Pull Request
SELECT 
  u.login AS user,
  p.name AS project,
  prh.actor_id AS developer_id,
  pr.base_repo_id AS project_id,
  'pull_request' AS interaction_type,
  prh.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.pull_request_history` prh
JOIN `ghtorrentmysql1906.MySQL1906.pull_requests` pr ON prh.pull_request_id = pr.id
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON prh.actor_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON pr.base_repo_id = p.id
WHERE prh.action = 'opened'
  AND EXTRACT(YEAR FROM prh.created_at) BETWEEN 2010 AND 2015

UNION ALL

-- Pull Request Comment
SELECT 
  u.login AS user,
  p.name AS project,
  prc.user_id AS developer_id,
  pr.base_repo_id AS project_id,
  'pull_comment' AS interaction_type,
  prc.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.pull_request_comments` prc
JOIN `ghtorrentmysql1906.MySQL1906.pull_requests` pr ON prc.pull_request_id = pr.id
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON prc.user_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON pr.base_repo_id = p.id
WHERE EXTRACT(YEAR FROM prc.created_at) BETWEEN 2010 AND 2015

UNION ALL

-- Commit
SELECT 
  u.login AS user,
  p.name AS project,
  c.author_id AS developer_id,
  c.project_id AS project_id,
  'commit' AS interaction_type,
  c.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.commits` c
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON c.author_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON c.project_id = p.id
WHERE EXTRACT(YEAR FROM c.created_at) BETWEEN 2010 AND 2015

UNION ALL

-- Follow (developer follows another developer)
SELECT 
  fu.login AS user,            -- follower
  tu.login AS project,         -- followed (kept as "project" for schema consistency)
  f.follower_id AS developer_id,
  f.user_id AS project_id,
  'follow' AS interaction_type,
  f.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.followers` f
JOIN `ghtorrentmysql1906.MySQL1906.users` fu ON f.follower_id = fu.id
JOIN `ghtorrentmysql1906.MySQL1906.users` tu ON f.user_id = tu.id
WHERE EXTRACT(YEAR FROM f.created_at) BETWEEN 2010 AND 2015
)
SELECT 
  i.*,
  p.owner_id AS project_owner_id,
  p.language AS project_language,
  p.description,
  u.country_code,
  u.location,
  u.company,
  u.type AS user_type_raw,
  u.fake = 0 AS is_real_user
FROM interactions i
LEFT JOIN `ghtorrentmysql1906.MySQL1906.projects` p
  ON i.project_id = p.id
LEFT JOIN `ghtorrentmysql1906.MySQL1906.users` u
  ON i.developer_id = u.id

WHERE i.interaction_time IS NOT NULL
  AND i.project_id IS NOT NULL
  AND i.developer_id IS NOT NULL
  AND p.owner_id IS NOT NULL
  AND p.language IS NOT NULL
  AND u.country_code IS NOT NULL
  AND u.location IS NOT NULL
  AND u.company IS NOT NULL
  AND u.type IS NOT NULL
  AND u.fake IS NOT NULL
  AND p.description IS NOT NULL
  
"""



# Set up BigQuery client
client = bigquery.Client()

# OPTIONAL: change this to your real dataset
destination_table = "gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015"

# Set query job config to write results to destination table
job_config = bigquery.QueryJobConfig(
    destination=destination_table,
    write_disposition="WRITE_TRUNCATE"  # Overwrite if table exists
)

# Run the query and save results to destination table
query_job = client.query(query, job_config=job_config, location="US")
query_job.result()  # Wait for job to complete

# # Now load the results from the table into DataFrame
# data_2008to2015 = client.query(f"SELECT * FROM `{destination_table}`").to_dataframe()

# # Output results
# print(data_2008to2015.head())


In [53]:
query = """

WITH interactions AS (
-- Star (same as watch, but included separately if needed)
SELECT 
  u.login AS user,
  p.name AS project,
  w.user_id AS developer_id,
  w.repo_id AS project_id,
  'star' AS interaction_type,
  w.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.watchers` w
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON w.user_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON w.repo_id = p.id
WHERE EXTRACT(YEAR FROM w.created_at) BETWEEN 2016 AND 2018


UNION ALL

-- Fork
SELECT 
  u.login AS user,
  p.name AS project,
  p.owner_id AS developer_id,
  p.forked_from AS project_id,
  'fork' AS interaction_type,
  p.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.projects` p
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON p.owner_id = u.id
WHERE p.forked_from IS NOT NULL
  AND EXTRACT(YEAR FROM p.created_at) BETWEEN 2016 AND 2018

UNION ALL

-- Issue
SELECT 
  u.login AS user,
  p.name AS project,
  i.reporter_id AS developer_id,
  i.repo_id AS project_id,
  'issue' AS interaction_type,
  i.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.issues` i
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON i.reporter_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON i.repo_id = p.id
WHERE EXTRACT(YEAR FROM i.created_at) BETWEEN 2016 AND 2018

UNION ALL

-- Pull Request
SELECT 
  u.login AS user,
  p.name AS project,
  prh.actor_id AS developer_id,
  pr.base_repo_id AS project_id,
  'pull_request' AS interaction_type,
  prh.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.pull_request_history` prh
JOIN `ghtorrentmysql1906.MySQL1906.pull_requests` pr ON prh.pull_request_id = pr.id
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON prh.actor_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON pr.base_repo_id = p.id
WHERE prh.action = 'opened'
  AND EXTRACT(YEAR FROM prh.created_at) BETWEEN 2016 AND 2018

UNION ALL

-- Pull Request Comment
SELECT 
  u.login AS user,
  p.name AS project,
  prc.user_id AS developer_id,
  pr.base_repo_id AS project_id,
  'pull_comment' AS interaction_type,
  prc.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.pull_request_comments` prc
JOIN `ghtorrentmysql1906.MySQL1906.pull_requests` pr ON prc.pull_request_id = pr.id
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON prc.user_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON pr.base_repo_id = p.id
WHERE EXTRACT(YEAR FROM prc.created_at) BETWEEN 2016 AND 2018

UNION ALL

-- Commit
SELECT 
  u.login AS user,
  p.name AS project,
  c.author_id AS developer_id,
  c.project_id AS project_id,
  'commit' AS interaction_type,
  c.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.commits` c
JOIN `ghtorrentmysql1906.MySQL1906.users` u ON c.author_id = u.id
JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON c.project_id = p.id
WHERE EXTRACT(YEAR FROM c.created_at) BETWEEN 2016 AND 2018

UNION ALL

-- Follow (developer follows another developer)
SELECT 
  fu.login AS user,            -- follower
  tu.login AS project,         -- followed (kept as "project" for schema consistency)
  f.follower_id AS developer_id,
  f.user_id AS project_id,
  'follow' AS interaction_type,
  f.created_at AS interaction_time
FROM `ghtorrentmysql1906.MySQL1906.followers` f
JOIN `ghtorrentmysql1906.MySQL1906.users` fu ON f.follower_id = fu.id
JOIN `ghtorrentmysql1906.MySQL1906.users` tu ON f.user_id = tu.id
WHERE EXTRACT(YEAR FROM f.created_at) BETWEEN 2016 AND 2018
)
SELECT 
  i.*,
  p.owner_id AS project_owner_id,
  p.language AS project_language,
  p.description,
  u.country_code,
  u.location,
  u.company,
  u.type AS user_type_raw,
  u.fake = 0 AS is_real_user
FROM interactions i
LEFT JOIN `ghtorrentmysql1906.MySQL1906.projects` p
  ON i.project_id = p.id
LEFT JOIN `ghtorrentmysql1906.MySQL1906.users` u
  ON i.developer_id = u.id

WHERE i.interaction_time IS NOT NULL
  AND i.project_id IS NOT NULL
  AND i.developer_id IS NOT NULL
  AND p.owner_id IS NOT NULL
  AND p.language IS NOT NULL
  AND u.country_code IS NOT NULL
  AND u.location IS NOT NULL
  AND u.company IS NOT NULL
  AND u.type IS NOT NULL
  AND u.fake IS NOT NULL
  AND p.description IS NOT NULL
  


"""



# Set up BigQuery client
client = bigquery.Client()

# OPTIONAL: change this to your real dataset
destination_table = "gh-masterthesis-jasmine.data_2008to2015.NEWdata_2016to2018"

# Set query job config to write results to destination table
job_config = bigquery.QueryJobConfig(
    destination=destination_table,
    write_disposition="WRITE_TRUNCATE"  # Overwrite if table exists
)

# Run the query and save results to destination table
query_job = client.query(query, job_config=job_config, location="US")
query_job.result()  # Wait for job to complete

#### Get the data from Bigquery

In [3]:
interaction_types = ["star", "fork", "issue", "pull_request", "pull_comment", "commit", "follow"]
dfs = []

In [ ]:


for interaction in interaction_types:
    query = f"""
    SELECT *
    FROM `gh-masterthesis-jasmine.data_2008to2015.data_2016to2018`
    WHERE interaction_type = '{interaction}'
    LIMIT 10000
    """
    df = client.query(query).to_dataframe()
    dfs.append(df)

# 合併成一個 DataFrame
partial_data = pd.concat(dfs, ignore_index=True)

# 存成 parquet
partial_data.to_parquet("oss_interactval_sample2.parquet", index=False)
print("✅")


✅


In [14]:
from google.cloud import bigquery
import pandas as pd

# 初始化 BigQuery Client
client = bigquery.Client()

# 你的資料表位置
destination_table = "gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015"

# 查詢：每種 interaction_type 各取最多 50,000 筆
query = r"""


-- Step 1: 計算每個 project 的總互動數量
WITH project_interaction_count AS (
  SELECT 
    project_id,
    COUNT(*) AS interaction_count
  FROM `gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015`
  GROUP BY project_id
),

-- Step 2: 計算每個開發者的 followers 數
dev_followers AS (
  SELECT 
    user_id,
    COUNT(*) AS num_followers
  FROM `ghtorrentmysql1906.MySQL1906.followers`
  GROUP BY user_id
),

-- Step 3: 過濾出同時符合兩個條件的資料
filtered AS (
  SELECT d.*
  FROM `gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015` d
  JOIN project_interaction_count pic ON d.project_id = pic.project_id
  JOIN dev_followers df ON d.developer_id = df.user_id
  WHERE LOWER(project_language) IN ("javascript", "java", "python", "php", "ruby", "c", "c++", "c#")
    AND REGEXP_CONTAINS(LOWER(description), "^[a-z0-9 ,.!?'\"()\\-:;]+$")
    AND pic.interaction_count >= 100         -- 熱門專案門檻
    AND df.num_followers >= 10               -- 有影響力開發者門檻
)

-- Step 4: 隨機取樣每種 interaction_type 各 50000 筆
SELECT *
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY interaction_type ORDER BY RAND()) AS rn
  FROM filtered
)
WHERE rn <= 50000

"""


partial_data = client.query(query, location="US").to_dataframe()

# 執行查詢並轉成 DataFrame
#partial_data = client.query(query).to_dataframe()

# 存成 Parquet
partial_data.to_parquet("Final2_oss_interaction_train.parquet", index=False)

print("✅ 已儲存 NEWoss_interaction_val.parquet（每種 interaction_type 各取 50,000 筆）")


✅ 已儲存 NEWoss_interaction_val.parquet（每種 interaction_type 各取 50,000 筆）


In [19]:
# Define your dataset and table
dataset_id = "gh-masterthesis-jasmine.data_2008to2015"
table_id = f"{dataset_id}.final_2"

# Load the table into a Pandas DataFrame
query = f"SELECT * FROM `{table_id}`"
df = client.query(query).to_dataframe()


# Alternatively, save as a Parquet file for efficiency
df.to_parquet("oss_interact_final.parquet", index=False)


In [ ]:
from google.cloud import bigquery
import pandas as pd

# 初始化 BigQuery Client
client = bigquery.Client()

# 你的資料表位置
destination_table = "gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015"

# 查詢：每種 interaction_type 各取最多 50,000 筆
query = r"""
-- 技術能力（technical ability）
WITH technical_ability AS (
  SELECT
    i.developer_id,
    i.project_id,
    COUNT(DISTINCT p.language) AS tech_ability_score
  FROM `gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015` i
  JOIN `ghtorrentmysql1906.MySQL1906.projects` p
    ON i.project_id = p.id
  WHERE p.language IS NOT NULL
  GROUP BY i.developer_id, i.project_id
),

-- 使用者檔案相似度（同公司）
user_profile AS (

  SELECT
    i.developer_id,
    i.project_id,
    
    COUNT(*) AS profile_similarity_score
  FROM gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015 i
  JOIN `ghtorrentmysql1906.MySQL1906.users` u_dev
    ON i.developer_id = u_dev.id
  JOIN `ghtorrentmysql1906.MySQL1906.users` u_owner
    ON i.project_owner_id = u_owner.id
  WHERE u_dev.company = u_owner.company  -- Ensure developer and owner share the same company
  GROUP BY i.developer_id, i.project_id
),


-- 社交連結
social_tie AS (
  SELECT
    i.developer_id,
    i.project_id,
    COUNT(*) AS social_tie_score
  FROM `gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015` i
  JOIN `gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015` i2
    ON i.developer_id = i2.developer_id AND i.project_id != i2.project_id
  WHERE i.interaction_type IN ('commit', 'pull_request', 'issue')
  GROUP BY i.developer_id, i.project_id
),

-- 專案互動次數
project_interaction_count AS (
  SELECT 
    project_id,
    COUNT(*) AS interaction_count
  FROM `gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015`
  GROUP BY project_id
),

-- 開發者的追蹤者數量
dev_followers AS (
  SELECT 
    user_id,
    COUNT(*) AS num_followers
  FROM `ghtorrentmysql1906.MySQL1906.followers`
  GROUP BY user_id
),

-- 加入所有特徵的主要查詢
filtered AS (
  SELECT 
    d.*,
    st.social_tie_score,
    ta.tech_ability_score,
    up.profile_similarity_score,
    
  FROM `gh-masterthesis-jasmine.data_2008to2015.NEWdata_2010to2015` d
  JOIN project_interaction_count pic ON d.project_id = pic.project_id
  JOIN dev_followers df ON d.developer_id = df.user_id
  JOIN user_profile up ON d.developer_id = up.developer_id AND d.project_id = up.project_id
  LEFT JOIN social_tie st ON d.developer_id = st.developer_id AND d.project_id = st.project_id
  LEFT JOIN technical_ability ta ON d.developer_id = ta.developer_id AND d.project_id = ta.project_id
  JOIN `ghtorrentmysql1906.MySQL1906.projects` p ON d.project_id = p.id
  WHERE LOWER(p.language) IN ("javascript", "java", "python", "php", "ruby", "c", "c++", "c#")
    AND REGEXP_CONTAINS(LOWER(p.description), "^[a-z0-9 ,.!?'\"()\\-:;]+$")
    AND pic.interaction_count >= 100
    AND df.num_followers >= 10
)

-- 最終隨機抽樣每種 interaction_type 各最多 50000 筆
SELECT *
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY interaction_type ORDER BY RAND()) AS rn
  FROM filtered
)
WHERE rn <= 50000
  AND interaction_time IS NOT NULL
  AND project_id IS NOT NULL
  AND developer_id IS NOT NULL
  AND social_tie_score IS NOT NULL
  AND tech_ability_score IS NOT NULL
  AND profile_similarity_score IS NOT NULL


"""


partial_data = client.query(query, location="US").to_dataframe()

# 執行查詢並轉成 DataFrame
#partial_data = client.query(query).to_dataframe()

# 存成 Parquet
partial_data.to_parquet("oss_interaction_Feature.parquet", index=False)

print("✅ 已儲存 NEWoss_interaction_val.parquet（每種 interaction_type 各取 50,000 筆）")


In [4]:
query = f"""
    SELECT *
    FROM `gh-masterthesis-jasmine.data_2008to2015.validation 2`
    """
df = client.query(query).to_dataframe()


# 合併成一個 DataFrame
#partial_data = pd.concat(df, ignore_index=True)

# 存成 parquet
df.to_parquet("Validation_final.parquet", index=False)
print("✅")

✅


In [26]:
Final["item_idx"].unique()  # 總開發者數量


<IntegerArray>
[  227, 36489,  6186, 15233, 41748, 35148,    40, 27056,  1798, 15585,
 ...
  6713, 15314, 10498, 14251,  2814, 30275, 19303,  3680, 39532,  8434]
Length: 49125, dtype: Int64

In [29]:
developer_project_counts = Final.groupby('user_idx')['project_id'].nunique().reset_index()

# Rename columns
developer_project_counts.columns = ['user_idx', 'num_projects']

developer_project_counts

,user_idx,num_projects
0,0,9
1,1,5
2,2,2
3,3,2
4,4,19
...,...,...
62542,62542,4
62543,62543,2
62544,62544,4
62545,62545,1


In [48]:
developer_project_counts['num_projects'].describe()

count    62547.000000
mean         3.469103
std          4.990275
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max        293.000000
Name: num_projects, dtype: float64

In [49]:
Final['social_tie_score_1'].describe()

count    350000.0
mean     6.979043
std      14.07815
min           0.0
25%           0.0
50%           1.0
75%           8.0
max         233.0
Name: social_tie_score_1, dtype: Float64